# Silver — CRM Customer Info
Customer master data from the CRM.

`bronze.crm_cust_info` → `silver.crm_customers`

## Init

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType, DateType

CATALOG = "workspace"

## Read bronze table

In [ ]:
df = spark.table(f"{CATALOG}.bronze.crm_cust_info")

## Transformations

### Trim all string columns

In [ ]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Normalize coded values
Single-letter codes become readable labels. Unknown values become `n/a` so downstream joins/filters are predictable.

In [ ]:
df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(col("cst_marital_status")) == "S", "Single")
         .when(F.upper(col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(col("cst_gndr")) == "F", "Female")
         .when(F.upper(col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

### Drop records without a customer id
A customer without an id cannot be joined to anything.

In [ ]:
df = df.filter(col("cst_id").isNotNull())

### Rename to business-friendly names

In [ ]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Sanity check

In [ ]:
df.limit(10).display()

## Write silver table

In [ ]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.silver.crm_customers")

In [ ]:
%sql
SELECT * FROM workspace.silver.crm_customers LIMIT 10;